# Feature Selection for Formula 1 Race Results

We want to predict which group a driver will finish in: **win** (1st), **podium** (2nd or 3rd), **points** (4th to 10th), or **rest** (11th or lower). Feature selection helps us choose the columns that are useful for this prediction.

We make our prediction just before the race starts. At that point we know the driver's starting position (`grid`). We only learn their final race status (`status`) after the race. Using historical training data, we can check how each pre-race feature relates to the finishing group before training a model.

This descriptive analysis pools train and validation. The validation outcomes now inform the rankings. Use a separate later period to evaluate any subset chosen here.

In [2]:
import pandas as pd
from scipy.stats import chi2_contingency
from sklearn.metrics import adjusted_mutual_info_score

In [2]:
# Check both files before combining them
columns = pd.read_csv("data/train.csv", nrows=0).columns.tolist()

assert set(columns) == set(pd.read_csv("data/validation.csv", nrows=0).columns)

print(f"Number of columns: {len(columns)}")
print(", ".join(columns))

# Group columns by what they describe
groups = {
    "Result identifier": [
        "resultId"
    ],
    "Race and circuit": [
        "racerId", "year", "round", "circuitId", "grand_prix",
        "date", "time_y", "url_x"
    ],
    "Practice, qualifying and sprint schedule": [
        "fp1_date", "fp1_time", "fp2_date", "fp2_time",
        "fp3_date", "fp3_time", "quali_date", "quali_time",
        "sprint_date", "sprint_time"
    ],
    "Driver": [
        "driverId", "driverRef", "forename", "surname", "dob",
        "nationality", "driver_num", "driver_code", "url_y"
    ],
    "Constructor (team)": [
        "constructorId", "constructorRef", "company", "nationality_y", "url"
    ],
    "Race entry and starting grid": [
        "number", "grid"
    ],
    "Race result and finishing status": [
        "positionOrder", "position_x", "positionText_x", "points",
        "laps", "time_x", "timetaken_in_millisec", "statusId", "status"
    ],
    "Fastest lap": [
        "fastestLap", "rank", "fastestLapTime", "max_speed"
    ],
    "Driver championship standings and related fields": [
        "driverStandingsId", "raceId_y", "points_y", "position",
        "positionText_y", "wins", "result_driver_standing"
    ]
}

# Show the columns found in the file and flag any that still need a group
grouped = set()

for group_name, members in groups.items():
    present = [col for col in members if col in columns]
    grouped.update(present)
    print(f"\n{group_name} ({len(present)}):")
    print(", ".join(present))

ungrouped = [col for col in columns if col not in grouped]
print(f"\nUngrouped columns: {ungrouped}")

Number of columns: 55
resultId, racerId, driverId, constructorId, number, grid, position_x, positionText_x, positionOrder, points, laps, time_x, timetaken_in_millisec, fastestLap, rank, fastestLapTime, max_speed, statusId, year, round, circuitId, grand_prix, date, time_y, url_x, fp1_date, fp1_time, fp2_date, fp2_time, fp3_date, fp3_time, quali_date, quali_time, sprint_date, sprint_time, driverRef, driver_num, driver_code, forename, surname, dob, nationality, url_y, driverStandingsId, raceId_y, points_y, position, positionText_y, wins, constructorRef, company, nationality_y, url, status, result_driver_standing

Result identifier (1):
resultId

Race and circuit (8):
racerId, year, round, circuitId, grand_prix, date, time_y, url_x

Practice, qualifying and sprint schedule (10):
fp1_date, fp1_time, fp2_date, fp2_time, fp3_date, fp3_time, quali_date, quali_time, sprint_date, sprint_time

Driver (9):
driverId, driverRef, forename, surname, dob, nationality, driver_num, driver_code, url_y

Co

### Decide how to use every source column

We predict immediately before the start. The training header is our checklist. We score pre-race categories directly, convert dates and times into interpretable features, and keep a reason for every exclusion. Alternative labels such as a driver's name are represented by the corresponding ID. Raw standings can come from the wrong round.

In [3]:
direct = ["grid", "driverId", "constructorId", "circuitId", "grand_prix", "year",
          "round", "number", "nationality", "nationality_y"]
sessions = ["fp1", "fp2", "fp3", "quali", "sprint"]

# Each source can contribute to more than one derived feature.
derived = {
    "month": ["date"],
    "weekday": ["date"],
    "driver_age_band": ["dob", "date"],
    "race_start_hour": ["time_y"]
}
for session in sessions:
    derived[session + "_days_before_race"] = [session + "_date", "date"]
    derived[session + "_hour"] = [session + "_time"]

excluded = {
    "Training label": ["positionOrder"],
    "Row or race key": ["resultId", "racerId"],
    "Known after the race": ["position_x", "positionText_x", "points", "laps", "time_x",
                             "timetaken_in_millisec", "fastestLap", "rank", "fastestLapTime",
                             "max_speed", "statusId", "status"],
    "Unreliable standings join": ["driverStandingsId", "raceId_y", "points_y", "position",
                                  "positionText_y", "wins", "result_driver_standing"],
    "Web link": ["url_x", "url_y", "url"],
    "Alternative driver or team label": ["forename", "surname", "driverRef", "driver_num",
                                         "driver_code", "constructorRef", "company"]
}
decisions = {col: reason for reason, names in excluded.items() for col in names}
for col in direct:
    decisions[col] = "Score directly"
for feature, sources in derived.items():
    for col in sources:
        decisions.setdefault(col, "Transform before scoring")

assert set(decisions) == set(columns), "Review the decisions against the CSV header."
audit = pd.DataFrame({"Source column": columns, "Decision": [decisions[col] for col in columns]})
audit["Derived features"] = [", ".join(f for f, sources in derived.items() if col in sources)
                             for col in columns]
with pd.option_context("display.max_rows", None):
    display(audit)

,Source column,Decision,Derived features
0,resultId,Row or race key,
1,racerId,Row or race key,
2,driverId,Score directly,
3,constructorId,Score directly,
4,number,Score directly,
5,grid,Score directly,
6,position_x,Known after the race,
7,positionText_x,Known after the race,
8,positionOrder,Training label,
9,points,Known after the race,


### Pool train and validation, keeping one row per driver result

First check all 55 original columns for any recorded value across both files. For ranking, keep the pre-race fields and the label only. The same result can appear with many different standings snapshots. Removing the snapshot columns lets us collapse those copies without choosing an arbitrary championship record.

In [4]:
source_cols = set(direct)
for sources in derived.values():
    source_cols.update(sources)
needed = sorted(source_cols | {"resultId", "positionOrder"})
parts = []
ids = {}
raw_rows = {}
has_value = pd.Series(False, index=columns)

for split in ["train", "validation"]:
    ids[split] = set()
    raw_rows[split] = 0
    for chunk in pd.read_csv(f"data/{split}.csv", dtype=str, na_values=r"\N", chunksize=100_000):
        raw_rows[split] += len(chunk)
        ids[split].update(chunk["resultId"].unique())
        has_value |= chunk.notna().any()
        parts.append(chunk[needed].drop_duplicates())

df = pd.concat(parts, ignore_index=True).drop_duplicates().reset_index(drop=True)
if df["resultId"].isna().any() or df["resultId"].duplicated().any():
    raise ValueError("Missing result ID or conflicting copies of a driver result.")
position = pd.to_numeric(df["positionOrder"], errors="raise")
if position.isna().any() or (position < 1).any() or (position % 1 != 0).any():
    raise ValueError("Invalid finishing position.")
target = pd.cut(position, [0, 1, 3, 10, float("inf")], labels=["win", "podium", "points", "rest"])
for split in ids:
    print(f"{split}: {raw_rows[split]:,} raw rows, {len(ids[split]):,} unique results")
overlap = ids["train"] & ids["validation"]
assert len(df) == len(ids["train"] | ids["validation"])
print(f"Results in both files: {len(overlap):,}")
print(f"Pooled results: {len(df):,}; redundant rows removed: {sum(raw_rows.values()) - len(df):,}")
empty_columns = has_value.index[~has_value].tolist()
print(f"Entirely empty source columns ({len(empty_columns)}): {', '.join(empty_columns)}")
display(target.value_counts(sort=False))

train: 2,830,101 raw rows, 22,089 unique results
validation: 353,762 raw rows, 1,836 unique results
Results in both files: 1
Pooled results: 23,924; redundant rows removed: 3,159,939
Entirely empty source columns (10): fp1_date, fp1_time, fp2_date, fp2_time, fp3_date, fp3_time, quali_date, quali_time, sprint_date, sprint_time


positionOrder
win         988
podium     1989
points     6942
rest      14005
Name: count, dtype: int64

### Prepare categories

Each recorded grid slot, racing number, round and season is a category. Race dates give month and weekday. Age is grouped into five-year bands (e.g. 25–29). Session dates give days before the race and times give the recorded hour. These schedule features describe the timetable; session results are excluded.

In [5]:
X = df[direct].copy()
race_date = pd.to_datetime(df["date"], errors="raise")
birth_date = pd.to_datetime(df["dob"], errors="raise")
X["month"] = race_date.dt.month
X["weekday"] = race_date.dt.day_name()
age = (race_date - birth_date).dt.days / 365.25
X["driver_age_band"] = (age // 5 * 5).astype("Int64")
X["race_start_hour"] = df["time_y"].str.slice(0, 2)
for session in sessions:
    session_date = pd.to_datetime(df[session + "_date"], errors="raise")
    X[session + "_days_before_race"] = (race_date - session_date).dt.days
    X[session + "_hour"] = df[session + "_time"].str.slice(0, 2)

### Measure association with the finishing class

For each feature, corrected Cramér’s V measures how far the observed counts are from independence. AMI measures shared information after adjusting for chance. We rank features separately within each measure.

In [6]:
def corrected_cramers_v(feature, target):
    counts = pd.crosstab(feature, target)
    rows, cols = counts.shape
    n = counts.to_numpy().sum()

    if rows < 2 or cols < 2:
        return float("nan")

    chi2 = chi2_contingency(counts, correction=False)[0]
    phi2 = max(0, chi2 / n - (rows - 1) * (cols - 1) / (n - 1))

    rows_corrected = rows - (rows - 1) ** 2 / (n - 1)
    cols_corrected = cols - (cols - 1) ** 2 / (n - 1)
    denominator = min(rows_corrected - 1, cols_corrected - 1)

    if denominator <= 0:
        return float("nan")

    return (phi2 / denominator) ** 0.5

### Rank the candidates and report automatic exclusions

A constant feature gives the same information for every start. Two features that partition the starts identically have the same scores, even if they use different labels. We keep the first and identify the equivalent feature. Coverage shows how many actual values were observed before missing values were labelled.

In [7]:
scores = []
removed = []
partitions = {}
for name in X.columns:
    values = X[name]
    observed = values.notna().sum()
    categories = values.astype("string").fillna("(missing)")
    codes, levels = pd.factorize(categories)
    if len(levels) == 1:
        reason = "Entirely missing" if observed == 0 else "Constant"
        removed.append((name, observed, reason))
        continue
    key = codes.tobytes()
    if key in partitions:
        removed.append((name, observed, "Same groups as " + partitions[key]))
        continue
    partitions[key] = name
    scores.append({
        "Feature": name, "Groups": len(levels), "Observed": observed,
        "Missing (%)": 100 * (1 - observed / len(values)),
        "Corrected V": corrected_cramers_v(categories, target),
        "AMI": adjusted_mutual_info_score(categories, target)
    })

ranking = pd.DataFrame(scores).set_index("Feature")
ranking["V rank"] = ranking["Corrected V"].rank(ascending=False)
ranking["AMI rank"] = ranking["AMI"].rank(ascending=False)
display(ranking.sort_values("V rank").round(4))
display(pd.DataFrame(removed, columns=["Excluded candidate", "Observed", "Reason"]))
print(f"{len(columns)} source columns reviewed; {len(X.columns)} candidate features; "
      f"{len(ranking)} ranked; {len(removed)} automatically excluded.")

,Groups,Observed,Missing (%),Corrected V,AMI,V rank,AMI rank
Feature,,,,,,,
grid,35,23924,0.0000,0.3940,0.0892,1.0,1.0
driverId,837,23924,0.0000,0.2867,0.0367,2.0,3.0
constructorId,207,23924,0.0000,0.2575,0.0439,3.0,2.0
number,129,23918,0.0251,0.2364,0.0321,4.0,4.0
nationality_y,24,23924,0.0000,0.1237,0.0198,5.0,5.0
nationality,41,23924,0.0000,0.1202,0.0129,6.0,6.0
year,69,23924,0.0000,0.0597,0.0021,7.0,8.0
driver_age_band,9,23924,0.0000,0.0478,0.0030,8.0,7.0
race_start_hour,18,5478,77.1025,0.0177,0.0004,9.0,9.0


,Excluded candidate,Observed,Reason
0,fp1_days_before_race,0,Entirely missing
1,fp1_hour,0,Entirely missing
2,fp2_days_before_race,0,Entirely missing
3,fp2_hour,0,Entirely missing
4,fp3_days_before_race,0,Entirely missing
5,fp3_hour,0,Entirely missing
6,quali_days_before_race,0,Entirely missing
7,quali_hour,0,Entirely missing
8,sprint_days_before_race,0,Entirely missing
9,sprint_hour,0,Entirely missing


55 source columns reviewed; 24 candidate features; 14 ranked; 10 automatically excluded.


Read each rank within its own metric. A feature with many missing values may mainly distinguish historical periods. A low individual score alone does not justify dropping a feature that might interact with the driver, team or grid. To measure a change in predictive performance, compare models on later races.

## Feature selection summary

Pooling `train.csv` and `validation.csv` and counting each `resultId` once gave us **23,924 driver starts**. We reviewed all **55 source columns** for a prediction made immediately before the race.

### Which columns did we keep?

| Decision | Count | Source columns | Reason and possible use |
|---|---:|---|---|
| Current race performance and outcome | 12 | `position_x`, `positionText_x`, `points`, `laps`, `time_x`, `timetaken_in_millisec`, `fastestLap`, `rank`, `fastestLapTime`, `max_speed`, `statusId`, `status` | These values become available after the start. Earlier races could provide a driver's recent retirement rate, for example. |
| Finishing position used to define the target | 1 | `positionOrder` | We use the final position to assign each start to win, podium, points or rest. Including it as a predictor would reveal the answer. |
| Entirely missing | 10 | `fp1_date`, `fp1_time`, `fp2_date`, `fp2_time`, `fp3_date`, `fp3_time`, `quali_date`, `quali_time`, `sprint_date`, `sprint_time` | Both files lack observations for these fields. Seasons with recorded schedules would allow us to assess them. |
| Record keys, links and alternative labels | 12 | `resultId`, `racerId`, `url_x`, `url_y`, `url`, `forename`, `surname`, `driverRef`, `driver_num`, `driver_code`, `constructorRef`, `company` | Record and race IDs help identify duplicates and group drivers by race. Links and names help identify records. Driver and constructor IDs already represent those identities in the ranking. |
| Championship standings from an uncertain round | 7 | `driverStandingsId`, `raceId_y`, `points_y`, `position`, `positionText_y`, `wins`, `result_driver_standing` | The same race result is linked to standings from multiple rounds. Linking each driver to the last completed round would allow us to assess points and wins before the start. |
| Used to build ranked features | 13 | `grid`, `driverId`, `constructorId`, `circuitId`, `grand_prix`, `year`, `round`, `number`, `nationality`, `nationality_y`, `date`, `dob`, `time_y` | We scored the first ten directly. Race date, birth date and start time supplied four more features: race month, weekday, driver age band and start hour. |

Of the 55 columns, 13 contain information available after the start (including the target), 10 are empty, 12 are keys, links or alternative labels, and 7 require a reliable standings record. This leaves **13 source columns**, which give **14 ranked features**.

### Results for all 14 features

Each score measures the association between one feature and the finishing class across the same 23,924 starts. Missing values form a separate category. The table is ordered by bias-corrected Cramér's V.

| Feature | Corrected Cramér's V | AMI |
|---|---:|---:|
| `grid` | 0.3940 | 0.0892 |
| `driverId` | 0.2867 | 0.0367 |
| `constructorId` | 0.2575 | 0.0439 |
| `number` | 0.2364 | 0.0321 |
| `nationality_y` | 0.1237 | 0.0198 |
| `nationality` | 0.1202 | 0.0129 |
| `year` | 0.0597 | 0.0021 |
| `driver_age_band` | 0.0478 | 0.0030 |
| `race_start_hour` | 0.0177 | 0.0004 |
| `round` | 0.0000 | -0.0004 |
| `circuitId` | 0.0000 | -0.0005 |
| `grand_prix` | 0.0000 | -0.0004 |
| `weekday` | 0.0000 | -0.0001 |
| `month` | 0.0000 | -0.0002 |

**Starting grid position ranked first by both measures.** The racing number ranked fourth in both. Comparing models with and without the racing number on later races would show whether it adds information beyond driver and team identity. Five features had a corrected V of zero when scored individually.

### What remains to be tested?

The first five rows of the decision table account for all **42 excluded source columns**. For `date`, `dob` and `time_y`, the scores refer to the derived features. Only 5,478 of 23,924 starts had a recorded start hour, so its score also reflects the pattern of missing data.

### Feature ranking by association with the finishing class

Both panels follow the order of corrected Cramér's V, with the strongest association at the top. Each measure has its own horizontal scale. AMI values just below zero indicate agreement slightly below the chance baseline.

In [ ]:
import matplotlib.pyplot as plt

ordered = ranking.sort_values("Corrected V", ascending=False, kind="stable")
fig, axes = plt.subplots(1, 2, figsize=(11, 6), sharey=True, layout="constrained")

for ax, metric, color in zip(axes, ["Corrected V", "AMI"], ["#236995", "#b45e29"]):
    ax.barh(ordered.index, ordered[metric], color=color)
    ax.set_title("Corrected Cramér's V" if metric == "Corrected V" else "Adjusted mutual information")
    ax.axvline(0, color="0.35", linewidth=0.8)
    ax.grid(axis="x", alpha=0.2)
    ax.set_axisbelow(True)

axes[0].invert_yaxis()
fig.suptitle("Which pre-race features are associated with the finishing class?")
plt.show()

In [4]:
test = pd.read_csv("data/test.csv")
test

C:\Users\mateu\AppData\Local\Temp\ipykernel_25884\1756180729.py:1: DtypeWarning: Columns (0: fastestLap, 1: max_speed) have mixed types. Specify dtype option on import or set low_memory=False.
  test = pd.read_csv("data/test.csv")


,resultId,racerId,driverId,constructorId,number,grid,position_x,positionText_x,positionOrder,points,...,driverStandingsId,raceId_y,points_y,wins,constructorRef,company,nationality_y,url,status,result_driver_standing
0,23949,997,4,1,14,13,8,8,8,4.0,...,12474,93,16.0,0,mclaren,McLaren,British,http://en.wikipedia.org/wiki/McLaren,+1 Lap,298739826
1,23949,997,4,1,14,13,8,8,8,4.0,...,12494,94,21.0,0,mclaren,McLaren,British,http://en.wikipedia.org/wiki/McLaren,+1 Lap,299218806
2,23949,997,4,1,14,13,8,8,8,4.0,...,12514,95,21.0,0,mclaren,McLaren,British,http://en.wikipedia.org/wiki/McLaren,+1 Lap,299697786
3,23949,997,4,1,14,13,8,8,8,4.0,...,12534,96,25.0,0,mclaren,McLaren,British,http://en.wikipedia.org/wiki/McLaren,+1 Lap,300176766
4,23949,997,4,1,14,13,8,8,8,4.0,...,12554,97,25.0,0,mclaren,McLaren,British,http://en.wikipedia.org/wiki/McLaren,+1 Lap,300655746
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
352923,26085,1110,857,1,81,5,\N,R,20,0.0,...,72085,1106,5.0,0,mclaren,McLaren,British,http://en.wikipedia.org/wiki/McLaren,Collision damage,1880337225
352924,26085,1110,857,1,81,5,\N,R,20,0.0,...,72125,1107,5.0,0,mclaren,McLaren,British,http://en.wikipedia.org/wiki/McLaren,Collision damage,1881380625
352925,26085,1110,857,1,81,5,\N,R,20,0.0,...,72145,1108,17.0,0,mclaren,McLaren,British,http://en.wikipedia.org/wiki/McLaren,Collision damage,1881902325
352926,26085,1110,857,1,81,5,\N,R,20,0.0,...,72165,1109,27.0,0,mclaren,McLaren,British,http://en.wikipedia.org/wiki/McLaren,Collision damage,1882424025
